# IMPORT LIBRARY

In [ ]:
import csv
import json
import os
import random
import re
import time
from urllib.robotparser import RobotFileParser
import requests
from bs4 import BeautifulSoup

# SCRAPING DATASET

In [ ]:
BASE = "https://filmindonesia.or.id"
CATALOG_URL = BASE + "/film?genre={slug}&page={page}"
USER_AGENT = "ProjekKuliah"   

# PENGATURAN
PRIORITAS = ["Horor", "Komedi", "Drama"]   # urutan prioritas label sekaligus urutan scraping
TARGET_PER_GENRE = 200
MODE_UJI = False           # True = percobaan kecil + verifikasi; False = scraping penuh
JUMLAH_UJI = 3             # film per kelas saat MODE_UJI
HEMAT_REQUEST = True       # False = selalu buka halaman detail tiap film (lebih lambat)
MAX_PAGES_PER_GENRE = 150  # batas pengaman halaman katalog per genre
MAX_GAGAL_BERUNTUN = 3     # berhenti jika film berturut-turut gagal dibuka (curiga diblokir)
MIN_SINOPSIS = 30          # sinopsis lebih pendek dari ini (karakter) dianggap kosong
DELAY = (2.5, 4.5)         # jeda acak antar request (detik), dihitung dari request terakhir
OUTPUT = "films_horor_komedi_drama.csv"
PROGRESS = "progres.jsonl"

FIELDS = ["judul", "sinopsis", "genre"]
RELEASE_RE = re.compile(r"^Tanggal rilis", re.I)
LABEL_RE = re.compile(r"^(Sutradara|Pemeran)\b", re.I)
ELLIPSIS = ("...", "…")

_terakhir = 0.0   # waktu request terakhir selesai
_jumlah_request = 0

def clean(text):
    """Rapikan spasi/baris baru berlebih."""
    return re.sub(r"\s+", " ", text or "").strip()

def tentukan_label(genres):
    """Satu label per film berdasarkan prioritas; None kalau bukan 3 kelas target."""
    punya = {g.lower() for g in genres}
    if "horor" in punya and "komedi" in punya:
        return None   # film horor-komedi tidak diambil (tidak jelas masuk kelas mana)
    for g in PRIORITAS:
        if g.lower() in punya:
            return g
    return None

def sudah_utuh(sinopsis):
    """Sinopsis katalog dianggap utuh kalau cukup panjang dan tidak berakhir '...'."""
    return len(sinopsis) >= MIN_SINOPSIS and not sinopsis.endswith(ELLIPSIS)

def parse_catalog(html):
    """Halaman katalog -> list kartu film: judul, daftar genre, sinopsis katalog, link detail."""
    soup = BeautifulSoup(html, "html.parser")
    cards = []
    for panel in soup.select("div.panel"):
        link = panel.select_one(".panel__heading__title a[href]")
        tags = panel.select_one(".panel__heading__tags")
        content = panel.select_one(".panel__content")
        if not (link and tags and content):   # lewati panel non-film (sidebar, dll.)
            continue
        genres = [
            clean(a.get_text())
            for span in tags.select('span[data-tippy-content="Genre"]')
            for a in span.select("a")
        ]
        sinopsis = ""
        for p in content.find_all("p"):
            t = clean(p.get_text(" "))
            if t and not RELEASE_RE.match(t) and not LABEL_RE.match(t):
                sinopsis = t
                break
        cards.append({
            "judul": clean(link.get_text()),
            "genres": [g for g in genres if g],
            "sinopsis": sinopsis,
            "url": link["href"],
        })
    return cards

def parse_detail(html):
    """Halaman detail film -> teks sinopsis lengkap ('' kalau tidak ada)."""
    soup = BeautifulSoup(html, "html.parser")
    for title in soup.select(".panel__heading__title"):
        if clean(title.get_text()).lower() == "sinopsis":
            panel = title.find_parent("div", class_="panel")
            content = panel.select_one(".panel__content") if panel else None
            if content:
                return clean(content.get_text(" "))
    return ""

def tunggu():
    """Diam secukupnya supaya jarak antar request sesuai DELAY (dihitung dari request terakhir)."""
    sisa = random.uniform(*DELAY) - (time.monotonic() - _terakhir)
    if sisa > 0:
        time.sleep(sisa)

def fetch(session, robots, url, retries=3):
    """Ambil satu halaman: cek robots.txt, jeda, ulangi kalau gagal."""
    global _terakhir, _jumlah_request
    if not robots.can_fetch(USER_AGENT, url):
        print("  dilarang oleh robots.txt:", url)
        return None
    for attempt in range(1, retries + 1):
        tunggu()
        _jumlah_request += 1
        try:
            r = session.get(url, timeout=20)
        except requests.RequestException as e:
            _terakhir = time.monotonic()
            print(f"  koneksi gagal ({e}), percobaan {attempt}/{retries}")
            time.sleep(5 * attempt)
            continue
        _terakhir = time.monotonic()
        if r.status_code == 200:
            return r.text
        if r.status_code in (500, 502, 504):
            print(f"  status {r.status_code} (error di server situs), percobaan {attempt}/{retries}")
            time.sleep(5 * attempt)
            continue
        if r.status_code in (403, 429, 503):
            print(f"  status {r.status_code}, menunggu sebelum mencoba lagi...")
            time.sleep(30 * attempt)
            continue
        print(f"  status {r.status_code}: {url}")
        return None
    return None

def cek_katalog(kat, det):
    """Mode uji: bandingkan sinopsis katalog dengan halaman detail, cetak hasilnya."""
    if not kat:
        hasil = ("detail ADA padahal katalog kosong -> PERINGATAN (set HEMAT_REQUEST = False)"
                 if det else "detail juga kosong -> aman dilewati")
    elif sudah_utuh(kat):
        hasil = ("sama dengan detail -> aman" if kat == det else
                 "BEDA dengan detail -> PERINGATAN (set HEMAT_REQUEST = False)")
    else:
        awal = kat.rstrip(".… ").strip()
        hasil = ("terpotong, detail memuat awalnya -> wajar" if det.startswith(awal) else
                 "terpotong tapi awalnya tidak cocok dengan detail -> PERINGATAN")
    print(f"      [cek] katalog: {hasil}")

def dapatkan_sinopsis(session, robots, card):
    """-> (sinopsis, status); status: 'katalog' | 'detail' | 'kosong' | 'gagal'."""
    kat = card["sinopsis"]
    if HEMAT_REQUEST and not MODE_UJI:
        if not kat:
            return "", "kosong"
        if sudah_utuh(kat):
            return kat, "katalog"
    html = fetch(session, robots, card["url"])
    if html is None:
        return "", "gagal"
    det = parse_detail(html)
    if HEMAT_REQUEST and MODE_UJI:
        cek_katalog(kat, det)
    return det, ("detail" if len(det) >= MIN_SINOPSIS else "kosong")

def load_progress():
    """-> (url sudah diproses, jumlah film per kelas, halaman terakhir yang tuntas per kelas)."""
    ada_csv, ada_prog = os.path.exists(OUTPUT), os.path.exists(PROGRESS)
    if ada_csv != ada_prog:
        raise SystemExit(
            f"{OUTPUT} dan {PROGRESS} harus ada berpasangan, tapi salah satunya tidak ada. "
            f"Hapus yang tersisa supaya mulai dari nol tanpa duplikat.")
    done, counts, selesai = set(), {g: 0 for g in PRIORITAS}, {g: 0 for g in PRIORITAS}
    if ada_prog:
        with open(PROGRESS, encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                except ValueError:
                    continue   # baris terakhir bisa rusak kalau proses terhenti mendadak
                g = r.get("genre")
                if g not in counts:
                    continue
                if "url" in r:
                    done.add(r["url"])
                    counts[g] += bool(r.get("ok"))
                elif "halaman_selesai" in r:
                    selesai[g] = max(selesai[g], r["halaman_selesai"])
    return done, counts, selesai

def main():
    global DELAY
    target = JUMLAH_UJI if MODE_UJI else TARGET_PER_GENRE
    print(f"Mode {'UJI' if MODE_UJI else 'PENUH'}: target {target} film per kelas "
          f"({' > '.join(PRIORITAS)}), HEMAT_REQUEST={HEMAT_REQUEST}.")

    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    robots = RobotFileParser()
    try:
        resp = session.get(BASE + "/robots.txt", timeout=20)
        robots.parse(resp.text.splitlines())
    except requests.RequestException as e:
        raise SystemExit(f"Tidak bisa membaca robots.txt ({e}). Coba lagi nanti.")
    crawl_delay = robots.crawl_delay(USER_AGENT)
    if crawl_delay:   # patuhi permintaan situs kalau ada
        DELAY = (max(DELAY[0], crawl_delay), max(DELAY[1], crawl_delay + 1))

    done, counts, selesai = load_progress()
    new_file = not os.path.exists(OUTPUT)
    gagal = False
    gagal_beruntun = 0
    dari = {"katalog": 0, "detail": 0, "kosong": 0}

    with open(OUTPUT, "a", encoding="utf-8", newline="") as f, \
         open(PROGRESS, "a", encoding="utf-8") as fp:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if new_file:
            writer.writeheader()

        def catat(rec):
            fp.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fp.flush()

        for genre in PRIORITAS:
            if counts[genre] >= target:
                print(f"\n=== {genre}: sudah {counts[genre]}/{target}, dilewati ===")
                continue
            slug = genre.lower().replace(" ", "-")
            mulai = selesai[genre] + 1
            print(f"\n=== {genre}: {counts[genre]}/{target}, mulai dari halaman {mulai} ===")
            prev_sig = None

            for page in range(mulai, MAX_PAGES_PER_GENRE + 1):
                print(f"Katalog {genre} halaman {page} ...")
                html = fetch(session, robots, CATALOG_URL.format(slug=slug, page=page))
                if html is None:
                    gagal = True
                    print("gagal membuka katalog, berhenti. Jalankan lagi nanti.")
                    break

                cards = parse_catalog(html)
                if not cards:
                    print("  tidak ada film (mungkin halaman terakhir).")
                    break
                if page == 1 and not any(genre.lower() in {x.lower() for x in c["genres"]}
                                         for c in cards):
                    print(f"  PERINGATAN: tidak ada film bergenre {genre} di halaman 1. "
                          f"Cek apakah filter '?genre={slug}' benar.")
                    break
                sig = [c["url"] for c in cards[:3]]
                if sig == prev_sig:
                    print("  isinya sama dengan halaman sebelumnya, berhenti.")
                    break
                prev_sig = sig

                for card in cards:
                    if counts[genre] >= target:
                        break
                    if card["url"] in done or tentukan_label(card["genres"]) != genre:
                        continue   # sudah diproses, atau labelnya kelas lain

                    print(f"  - {card['judul']}", flush=True)
                    sinopsis, status = dapatkan_sinopsis(session, robots, card)
                    if status == "gagal":
                        gagal_beruntun += 1
                        if gagal_beruntun >= MAX_GAGAL_BERUNTUN:
                            gagal = True
                            print(f"    gagal {gagal_beruntun}x berturut-turut, berhenti "
                                  f"(mungkin diblokir). Jalankan lagi nanti.")
                            break
                        print("    gagal membuka detail, film ini dilewati")
                        catat({"url": card["url"], "genre": genre, "ok": False})
                        done.add(card["url"])
                        continue
                    gagal_beruntun = 0

                    ok = status in ("katalog", "detail")
                    dari[status] += 1
                    if ok:
                        writer.writerow({"judul": card["judul"], "sinopsis": sinopsis,
                                         "genre": genre})
                        f.flush()
                        counts[genre] += 1
                        print(f"    ok [{counts[genre]}/{target}] (sinopsis dari {status})")
                    else:
                        print("    dilewati (sinopsis kosong)")
                    catat({"url": card["url"], "genre": genre, "ok": ok})
                    done.add(card["url"])

                if gagal or counts[genre] >= target:
                    break   # halaman ini mungkin belum tuntas, jadi tidak ditandai selesai
                catat({"genre": genre, "halaman_selesai": page})

            if gagal:
                break

    status_akhir = "BERHENTI karena gagal (jalankan lagi untuk melanjutkan)" if gagal else "Selesai"
    print(f"\n{status_akhir}. Film per kelas: {counts} -> {OUTPUT}")
    print(f"Request ke situs: {_jumlah_request}. Sinopsis dipakai dari katalog: "
          f"{dari['katalog']}, dari halaman detail: {dari['detail']}, dilewati: {dari['kosong']}.")

if __name__ == "__main__":
    main()